In [ ]:
#| default_exp train_hmep

# train_hmep

Hierarchical MEP (HMEP): trains a `SwinMaskedEmbeddingPredictor` to predict fine-level
embeddings (L4, L5) from coarse-level embeddings (L0-L3) only.

This is a post-encoder training step. The encoder is frozen. HMEP is initialized from
the encoder's MEP checkpoint (which already learned cross-level structure), then
fine-tuned for the specific task of coarse→fine prediction.

Pipeline position: train_enc → preencode → **train_hmep** → train_dec

The trained HMEP enables the generative pipeline:
  flow model generates L0-L3 → HMEP predicts L4-L5 → decoder reconstructs image

In [ ]:
#| export
import os, sys
sys.dont_write_bytecode = True
os.environ['WANDB_QUIET'] = 'true'
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import wandb
import hydra
from hydra.core.hydra_config import HydraConfig
from omegaconf import DictConfig

from midi_rae.core import *
from midi_rae.swin import SwinEncoder, SwinMaskedEmbeddingPredictor
from midi_rae.data import AnchorDataset
from midi_rae.utils import *

In [ ]:
#| export
from midi_rae.data import (PreEncodedChunkDataset, ChunkShuffleSampler,
                            collate_emb_levels, get_pos_cache, emb_levels_to_enc_out)


In [ ]:
#| export
def mask_fine_levels(enc_out, n_fine=2):
    """Zero out the finest n_fine levels in enc_out, returning masked enc_out and original fine embeddings.
    Input to HMEP has fine levels zeroed; targets are the original fine embeddings."""
    levels = enc_out.patches.levels
    n = len(levels)
    fine_idx = list(range(n - n_fine, n))   # e.g. [4, 5] for n_fine=2
    targets = [levels[i].emb.detach() for i in fine_idx]
    new_levels = [
        PatchState(emb=torch.zeros_like(lvl.emb) if i in fine_idx else lvl.emb,
                   pos=lvl.pos, non_empty=lvl.non_empty, mae_mask=lvl.mae_mask)
        for i, lvl in enumerate(levels)
    ]
    masked_enc_out = EncoderOutput(
        patches=HierarchicalPatchState(levels=new_levels),
        full_pos=enc_out.full_pos, full_non_empty=enc_out.full_non_empty, mae_mask=enc_out.mae_mask)
    return masked_enc_out, targets, fine_idx

In [ ]:
#| export
def hmep_loss(hmep, enc_out, n_fine=2):
    """Forward HMEP on coarse-only input; return MSE loss against real fine embeddings.
    Returns (loss, per_level_losses dict)."""
    masked_enc_out, targets, fine_idx = mask_fine_levels(enc_out, n_fine=n_fine)
    preds, _masks = hmep(masked_enc_out, mask_ratio=0.0)  # mask_ratio=0: no additional random masking
    loss = torch.tensor(0.0, device=targets[0].device)
    per_level = {}
    for k, (fi, target) in enumerate(zip(fine_idx, targets)):
        lvl_loss = F.mse_loss(preds[fi], target)
        per_level[f'L{fi}'] = lvl_loss.item()
        loss = loss + lvl_loss
    return loss / len(fine_idx), per_level

In [ ]:
#| export
def train_hmep(cfg: DictConfig):
    if torch.cuda.is_available(): device = 'cuda'
    elif torch.backends.mps.is_available(): device = 'mps'
    else: device = 'cpu'
    cjprint(f'device={device}  config={HydraConfig.get().job.config_name}', color='green')
    set_seed()

    n_fine = cfg.training.get('n_fine_levels', 2)
    use_preencoded = cfg.get('use_preencoded', False)

    # --- Encoder (frozen) ---
    dims = [cfg.model.embed_dim * (2 ** (len(cfg.model.depths) - 1 - i))
            for i in range(len(cfg.model.depths))]
    encoder = SwinEncoder(
        img_height=cfg.data.image_size, img_width=cfg.data.image_size,
        patch_h=cfg.model.patch_h, patch_w=cfg.model.patch_w,
        embed_dim=cfg.model.embed_dim, depths=cfg.model.depths,
        num_heads=cfg.model.num_heads, window_size=cfg.model.window_size,
        mlp_ratio=cfg.model.mlp_ratio, drop_path_rate=cfg.model.drop_path_rate,
    ).to(device)
    encoder = load_checkpoint(encoder, cfg.encoder_ckpt)
    encoder.eval()
    for p in encoder.parameters(): p.requires_grad = False
    cjprint('Encoder frozen ❄️', color='blue')

    pos_cache = get_pos_cache(encoder, cfg.data.image_size, device)

    # --- HMEP ---
    hmep = SwinMaskedEmbeddingPredictor(dims=dims, n_summaries=(1,2,4,8,32,64)).to(device)
    hmep_init_ckpt = cfg.get('hmep_init_ckpt', None)
    if hmep_init_ckpt:
        hmep = load_checkpoint(hmep, hmep_init_ckpt)
        cjprint(f'HMEP initialized from {hmep_init_ckpt}', color='cyan')
    else:
        cjprint('HMEP initialized randomly', color='yellow')

    # --- Data ---
    nw = cfg.training.num_workers
    if use_preencoded:
        encoded_dir = cfg.preencode.output_dir
        cjprint(f'Using pre-encoded embeddings from {encoded_dir}', color='cyan')
        train_ds = PreEncodedChunkDataset(encoded_dir, split='train', img_key=None)
        val_ds   = PreEncodedChunkDataset(encoded_dir, split='val', img_key=None)
        train_dl = DataLoader(train_ds, batch_size=cfg.training.batch_size,
                              sampler=ChunkShuffleSampler(train_ds, shuffle=True),
                              num_workers=nw[0], persistent_workers=(nw[0]>0),
                              collate_fn=collate_emb_levels)
        val_dl   = DataLoader(val_ds,   batch_size=cfg.training.batch_size,
                              sampler=ChunkShuffleSampler(val_ds, shuffle=False),
                              num_workers=nw[1], persistent_workers=(nw[1]>0),
                              collate_fn=collate_emb_levels)
    else:
        cjprint('Encoding on the fly', color='yellow')
        train_ds = AnchorDataset(image_dataset_dir=cfg.data.path, split='train',
                                 aug_y_max=cfg.training.max_shift_y)
        val_ds   = AnchorDataset(image_dataset_dir=cfg.data.path, split='val',
                                 aug_y_max=cfg.training.max_shift_y)
        train_dl = DataLoader(train_ds, batch_size=cfg.training.batch_size, shuffle=True,
                              num_workers=nw[0], pin_memory=(device=='cuda'),
                              persistent_workers=(nw[0]>0))
        val_dl   = DataLoader(val_ds,   batch_size=cfg.training.batch_size, shuffle=False,
                              num_workers=nw[1], pin_memory=(device=='cuda'),
                              persistent_workers=(nw[1]>0))

    # --- Optimizer ---
    epochs = cfg.training.get('hmep_epochs', 50)
    lr     = cfg.training.get('hmep_lr', 1e-4)
    steps_per_epoch = cfg.training.get('steps_per_epoch', None)
    opt    = torch.optim.AdamW(hmep.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, steps_per_epoch=steps_per_epoch or len(train_dl), epochs=epochs)

    if not cfg.get('no_wandb', False):
        wandb.init(project='hmep-'+cfg.wandb.project, config=dict(cfg))
        wandb.run.name = f"{cfg.tag}_{wandb.run.name}"

    best_val_loss = float('inf')
    global_step = 0
    for epoch in range(1, epochs + 1):
        hmep.train()
        train_loss = 0.0
        n_train = 0
        for i, batch in enumerate(tqdm(train_dl, desc=f'Epoch {epoch}/{epochs}',
                                       total=steps_per_epoch or len(train_dl))):
            if steps_per_epoch and i >= steps_per_epoch: break
            if use_preencoded:
                enc_out = emb_levels_to_enc_out(batch, pos_cache, device)
            else:
                img = batch['img'].to(device)
                with torch.no_grad():
                    enc_out = encoder(img)
            opt.zero_grad()
            loss, _ = hmep_loss(hmep, enc_out, n_fine=n_fine)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(hmep.parameters(), 1.0)
            opt.step()
            scheduler.step()
            train_loss += loss.item()
            n_train += 1
            global_step += 1
            if wandb.run is not None:
                wandb.log({'train_loss_step': loss.item(), 'lr': scheduler.get_last_lr()[0]}, step=global_step)
        train_loss /= n_train

        hmep.eval()
        val_loss, val_per_level = 0.0, {}
        with torch.no_grad():
            for batch in val_dl:
                if use_preencoded:
                    enc_out = emb_levels_to_enc_out(batch, pos_cache, device)
                else:
                    img = batch['img'].to(device)
                    enc_out = encoder(img)
                loss, per_level = hmep_loss(hmep, enc_out, n_fine=n_fine)
                val_loss += loss.item()
                for k, v in per_level.items():
                    val_per_level[k] = val_per_level.get(k, 0) + v
        val_loss /= len(val_dl)
        val_per_level = {k: v/len(val_dl) for k, v in val_per_level.items()}

        if val_loss < best_val_loss: best_val_loss = val_loss
        per_lvl_str = '  '.join(f'{k}={v:.4f}' for k, v in val_per_level.items())
        print(f'Epoch {epoch}: train={train_loss:.6f}  val={val_loss:.6f}  best={best_val_loss:.6f}  [{per_lvl_str}]')

        if wandb.run is not None:
            wandb.log({'train_loss': train_loss, 'val_loss': val_loss, 'epoch': epoch,
                       **{f'val_{k}': v for k, v in val_per_level.items()}}, step=global_step)

        save_checkpoint(hmep, epoch, val_loss, cfg, optimizer=opt, tag=cfg.tag)

    wandb.finish()
    return best_val_loss

In [ ]:
#| export
#| eval: false
@hydra.main(version_base=None, config_path='../configs', config_name='config')
def train_hmep_main(cfg: DictConfig):
    cjprint('Hierarchical MEP training: predict fine levels from coarse', color='cyan')
    best_metric = train_hmep(cfg)
    print(f'FINISHED. Best metric: {best_metric:.6f}')

if __name__ == '__main__' and 'ipykernel' not in __import__('sys').modules:
    train_hmep_main()